# Home Exercise 2 on Text Generation
Implement a sequence2sequence to summarize the text. 

Data: [CNN-DailyMail News Text Summarization](https://www.kaggle.com/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail) 

In [1]:
import os, shutil

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gowrishankarp/newspaper-text-summarization-cnn-dailymail")

print("Path to dataset files:", path)

Path to dataset files: /home/dikhang_hcmut/.cache/kagglehub/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/versions/2


In [3]:
import zipfile
def move_path(src_path: str, dest_dir: str):
    if not os.path.exists(src_path):
        raise FileNotFoundError(f"Source file not found: {src_path}")
    
    os.makedirs(dest_dir, exist_ok=True)
    dst_path = os.path.join(dest_dir, os.path.basename(src_path))
    
    if os.path.exists(dst_path):
        print(f"Destination {dst_path} exists. Overwriting...")
        if os.path.isdir(dst_path):
            shutil.rmtree(dst_path)
        else:
            os.remove(dst_path)

    # Thực hiện di chuyển
    new_path = shutil.move(src_path, dest_dir)
    return new_path

def unzip(path, dest, delete=True):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Zip file does not exist: {path}")
    
    if not zipfile.is_zipfile(path):
        raise zipfile.BadZipFile(f"Not a valid zip file: {path}")
    
    print(f"Extracting {path} to {dest}...")
    with zipfile.ZipFile(path, 'r') as zip_ref:
        zip_ref.extractall(dest)
        print(f"Unzipped successfully into: {dest}")
    
    if delete:
        os.remove(path)
        print(f"Deleted zip file: {path}")
    else:
        print(f"Kept zip file.")
    
    return dest

In [4]:
!ls /home/dikhang_hcmut/.cache/kagglehub/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/versions/2/cnn_dailymail

test.csv  train.csv  validation.csv


In [5]:
# cache_root = "/home/dikhang/.cache/kagglehub/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/versions/2"
source_data_dir = os.path.join(path, "cnn_dailymail")

source_dir = "./data"
# Thư mục đích sẽ chứa file csv
data_dir = os.path.join(source_dir, "cnn_dailymail")

print(f"Source Directory: {source_data_dir}")
print(f"Destination Directory: {data_dir}")

os.makedirs(data_dir, exist_ok=True)

files = ["train.csv", "test.csv", "validation.csv"]

try:
    if not os.path.exists(source_data_dir):
        raise FileNotFoundError(f"Cannot found: {source_data_dir}")

    for file_name in files:
        src_file = os.path.join(source_data_dir, file_name)
        dst_file = os.path.join(data_dir, file_name)
        

        if os.path.exists(src_file):
            shutil.copy2(src_file, dst_file)
            print(f"Đã sao chép: {file_name}")
        else:
            print(f"Không tìm thấy file {file_name} trong nguồn.")
            
except Exception as e:
    print(f"Lỗi nghiêm trọng trong quá trình sao chép: {e}")
    exit(1)

train_path = os.path.join(data_dir, "train.csv")
test_path = os.path.join(data_dir, "test.csv")
val_path = os.path.join(data_dir, "validation.csv") 

print(f"Train path variable: {train_path}")
print(f"Val path variable:   {val_path}")

Source Directory: /home/dikhang_hcmut/.cache/kagglehub/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/versions/2/cnn_dailymail
Destination Directory: ./data/cnn_dailymail
Đã sao chép: train.csv
Đã sao chép: test.csv
Đã sao chép: validation.csv
Train path variable: ./data/cnn_dailymail/train.csv
Val path variable:   ./data/cnn_dailymail/validation.csv


## Data preparation

### Import libraries

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import spacy
import random
from datetime import datetime
from collections import Counter

print(f"The last time this notebook was run is: {datetime.now().strftime('%H:%M:%S %d/%m/%y')}")

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

The last time this notebook was run is: 09:24:45 15/12/25
Using device: cuda


## Configuration for training model

In [7]:
import torch
import os

class Config:
    ROOT = "./data/cnn_dailymail"
    TRAIN_PATH = os.path.join(ROOT, "train.csv")
    VAL_PATH = os.path.join(ROOT, "validation.csv")
    TEST_PATH = os.path.join(ROOT, "test.csv")
    VOCAB_PATH = os.path.join(ROOT, "vocab.pt")

    # Data Param
    MAX_LEN_SRC = 80
    MAX_LEN_TRG = 50
    MIN_FREQ = 3
    MAX_VOCAB = 30000

    BATCH_SIZE = 32
    ACCUM_STEPS = 2
    DEBUG = None 

    ENC_EMB_DIM = 256
    DEC_EMB_DIM = 256
    HID_DIM = 256
    N_LAYERS = 1
    ENC_DROPOUT = 0.3
    DEC_DROPOUT = 0.3
    
    # ======================
    # Training Param
    CLIP = 1.0
    LEARNING_RATE = 5e-4
    N_EPOCHS = 10

    TEACHER_FORCING_RATIO = 0.5

    # Early stopping
    PATIENCE = 2

    PAD_IDX, UNK_IDX, SOS_IDX, EOS_IDX = 0, 1, 2, 3
    TOKEN_MAP = {
        '<pad>': 0,
        '<unk>': 1,
        '<sos>': 2,
        '<eos>': 3
    }
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    INPUT_DIM = 0
    OUTPUT_DIM = 0


## Dataset class

In [8]:
import re
class CNNDailyMailDataset(Dataset):
    def __init__(self, csv_path, config, is_train=True):
        self.config = config
        
        # 1. Load Data
        if config.DEBUG:
            self.df = pd.read_csv(csv_path, nrows=config.DEBUG)
        else:
            self.df = pd.read_csv(csv_path)
        
        # 2. Xử lý Vocab
        if is_train:
            if os.path.exists(config.VOCAB_PATH) and not config.DEBUG:
                print(f"Loading vocab from: {config.VOCAB_PATH}")
                self.vocab = torch.load(config.VOCAB_PATH)
            else:
                self.vocab = self.build_vocab()
                if not config.DEBUG:
                    torch.save(self.vocab, config.VOCAB_PATH)
        else:
            if os.path.exists(config.VOCAB_PATH):
                self.vocab = torch.load(config.VOCAB_PATH)
            else:
                raise ValueError("Vocab not found. Run train first.")

        self.config.INPUT_DIM = len(self.vocab)
        self.config.OUTPUT_DIM = len(self.vocab)

    def tokenize(self, text):
        if not isinstance(text, str): return []
        text = text.lower()
        return re.findall(r"[\w']+|[.,!?;]", text)

    def build_vocab(self):
        print("Building vocab...")
        counter = Counter()
        total = len(self.df)
        for i, row in self.df.iterrows():
            if i % 10000 == 0: print(f"Processed {i}/{total} rows...", end='\r')
            
            counter.update(self.tokenize(row['article']))
            counter.update(self.tokenize(row['highlights']))
            
        vocab = self.config.TOKEN_MAP.copy()
        idx = 4
        for word, count in counter.most_common(self.config.MAX_VOCAB):
            if count >= self.config.MIN_FREQ:
                vocab[word] = idx
                idx += 1
        
        print(f"\nDone! Vocab size: {len(vocab)}")
        return vocab

    def text_to_indices(self, text, max_len):
        tokens = self.tokenize(text)[:max_len-2]
        indices = [self.config.SOS_IDX] + \
                  [self.vocab.get(t, self.config.UNK_IDX) for t in tokens] + \
                  [self.config.EOS_IDX]
        
        if len(indices) < max_len:
            indices += [self.config.PAD_IDX] * (max_len - len(indices))
        return torch.tensor(indices, dtype=torch.long)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        try:
            row = self.df.iloc[idx]
            src = self.text_to_indices(row['article'], self.config.MAX_LEN_SRC)
            trg = self.text_to_indices(row['highlights'], self.config.MAX_LEN_TRG)
            return src, trg
        except:
            return torch.zeros(self.config.MAX_LEN_SRC, dtype=torch.long), \
                   torch.zeros(self.config.MAX_LEN_TRG, dtype=torch.long)

## Loading the data

### Testing the dataset class

In [9]:
Config.DEBUG = 1000 
train_dataset = CNNDailyMailDataset(Config.TRAIN_PATH, Config, is_train=True)
train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)

batch = next(iter(train_loader))
print("Src:", batch[0].shape)
print("Trg:", batch[1].shape)

Building vocab...
Processed 0/1000 rows...
Done! Vocab size: 17006
Src: torch.Size([32, 80])
Trg: torch.Size([32, 50])


### Set the DEBUG flag to False for training the model

In [10]:
Config.DEBUG = None
train_dataset = CNNDailyMailDataset(Config.TRAIN_PATH, Config, is_train=True)
train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)

batch = next(iter(train_loader))
print("Src:", batch[0].shape)
print("Trg:", batch[1].shape)

Loading vocab from: ./data/cnn_dailymail/vocab.pt
Src: torch.Size([32, 80])
Trg: torch.Size([32, 50])


In [11]:
val_dataset = CNNDailyMailDataset(Config.VAL_PATH, Config, is_train=False)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)

val_batch = next(iter(val_loader))
print(f"Val Src: {val_batch[0].shape}")
print(f"Val Trg: {val_batch[1].shape}")

Val Src: torch.Size([32, 80])
Val Trg: torch.Size([32, 50])


## Model architecture

In [12]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, bidirectional=True)
        self.fc = nn.Linear(hid_dim * 2, hid_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        # src: [src_len, batch_size]
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        # hidden: [2, batch, hid_dim] -> [batch, hid_dim]
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear((hid_dim * 2) + hid_dim, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)
        
    def forward(self, hidden, encoder_outputs):
        # hidden: [batch, hid_dim]
        # encoder_outputs: [src_len, batch, hid_dim*2]
        batch_size = encoder_outputs.shape[1]
        src_len = encoder_outputs.shape[0]
        
        hidden = hidden.unsqueeze(1).expand(-1, src_len, -1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU((hid_dim * 2) + emb_dim, hid_dim)
        self.fc_out = nn.Linear((hid_dim * 2) + hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, encoder_outputs):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        weighted = torch.bmm(a, encoder_outputs).permute(1, 0, 2)
        
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        
        embedded = embedded.squeeze(0)
        output = output.squeeze(0)
        weighted = weighted.squeeze(0)
        
        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))
        return prediction, hidden.squeeze(0)

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        encoder_outputs, hidden = self.encoder(src)
        
        input = trg[0,:]
        for t in range(1, trg_len):
            output, hidden = self.decoder(input, hidden, encoder_outputs)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1) 
            input = trg[t] if teacher_force else top1
            
        return outputs

In [13]:
def init_weights(m):
    if isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, mean=0, std=0.1)

    elif isinstance(m, nn.GRU):
        for name, param in m.named_parameters():
            if "weight" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.constant_(param, 0)

    elif isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)


scaler = torch.amp.GradScaler("cuda")

def train_epoch(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    ACCUM_STEPS = Config.ACCUM_STEPS
    for i, (src, trg) in enumerate(iterator):
        src, trg = src.to(Config.DEVICE), trg.to(Config.DEVICE)
        src, trg = src.permute(1, 0), trg.permute(1, 0)

        with torch.amp.autocast("cuda"):
            output = model(src, trg)
            output_dim = output.shape[-1]
            output = output[1:].reshape(-1, output_dim)
            trg = trg[1:].reshape(-1)
            loss = criterion(output, trg) / ACCUM_STEPS
        if i % 100 == 0:
            print(f"Batch {i}/{len(iterator)} | loss={loss.item():.3f}")
        scaler.scale(loss).backward()

        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        epoch_loss += loss.item() * ACCUM_STEPS

    return epoch_loss / len(iterator)



def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, (src, trg) in enumerate(iterator):
            src, trg = src.to(Config.DEVICE), trg.to(Config.DEVICE)
            src, trg = src.permute(1, 0), trg.permute(1, 0)

            output = model(src, trg, 0)
            output_dim = output.shape[-1]
            output = output[1:].reshape(-1, output_dim)
            trg = trg[1:].reshape(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(iterator)

### Training loop

In [ ]:
import time

# Init Components
attn = Attention(Config.HID_DIM)
enc = Encoder(Config.INPUT_DIM, Config.ENC_EMB_DIM, Config.HID_DIM, Config.ENC_DROPOUT)
dec = Decoder(Config.OUTPUT_DIM, Config.DEC_EMB_DIM, Config.HID_DIM, Config.DEC_DROPOUT, attn)
model = Seq2Seq(enc, dec, Config.DEVICE).to(Config.DEVICE)

model.apply(init_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=Config.PAD_IDX)

best_valid_loss = float('inf')
train_losses, valid_losses = [], []
EPOCHS = Config.N_EPOCHS if not Config.DEBUG else 2

torch.backends.cudnn.benchmark = True

print("Src:", batch[0].shape)
print("Trg:", batch[1].shape)
print("Start Training...")
for epoch in range(EPOCHS):
    start_time = time.time()

    train_loss = train_epoch(model, train_loader, optimizer, criterion, Config.CLIP)
    valid_loss = evaluate(model, val_loader, criterion)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    end_time = time.time()
    mins, secs = divmod(end_time - start_time, 60)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(Config.ROOT, 'best_model.pt'))
        msg = "(Saved)"
    else:
        patience_counter += 1
        msg = f"(No improve {patience_counter}/{Config.PATIENCE})"

    print(f'Epoch: {epoch+1:02} | Time: {int(mins)}m {int(secs)}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Val Loss: {valid_loss:.3f} {msg}')

    if patience_counter >= Config.PATIENCE:
        print("⏹ Early stopping triggered.")
        break


Src: torch.Size([32, 80])
Trg: torch.Size([32, 50])
Start Training...
Batch 0/8973 | loss=5.155
Batch 100/8973 | loss=3.580
Batch 200/8973 | loss=3.522
Batch 300/8973 | loss=3.486
Batch 400/8973 | loss=3.484
Batch 500/8973 | loss=3.485
Batch 600/8973 | loss=3.460
Batch 700/8973 | loss=3.410
Batch 800/8973 | loss=3.308
Batch 900/8973 | loss=3.392
Batch 1000/8973 | loss=3.350
Batch 1100/8973 | loss=3.326
Batch 1200/8973 | loss=3.323
Batch 1300/8973 | loss=3.319
Batch 1400/8973 | loss=3.378
Batch 1500/8973 | loss=3.341
Batch 1600/8973 | loss=3.332
Batch 1700/8973 | loss=3.247
Batch 1800/8973 | loss=3.222
Batch 1900/8973 | loss=3.154
Batch 2000/8973 | loss=3.187
Batch 2100/8973 | loss=3.268
Batch 2200/8973 | loss=3.108
Batch 2300/8973 | loss=3.251
Batch 2400/8973 | loss=3.195
Batch 2500/8973 | loss=3.167
Batch 2600/8973 | loss=3.167


In [ ]:
print("INPUT_DIM =", Config.INPUT_DIM)
print("OUTPUT_DIM =", Config.OUTPUT_DIM)
import gc

# Clear Python objects
gc.collect()

# Clear CUDA cache
torch.cuda.empty_cache()

# Reset CUDA peak memory stats
torch.cuda.reset_peak_memory_stats()

print("CUDA refreshed")

INPUT_DIM = 30004
OUTPUT_DIM = 30004
CUDA refreshed


In [ ]:
import matplotlib.pyplot as plt
def plot_learning_curves(train_losses, valid_losses):
    plt.figure(figsize=(10, 5))
    plt.title("Training and Validation Loss")
    plt.plot(train_losses, label="Train Loss", color='blue', linestyle='-')
    plt.plot(valid_losses, label="Validation Loss", color='orange', linestyle='--')
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_learning_curves(train_losses, valid_losses)

In [ ]:
import matplotlib.ticker as ticker
import seaborn as sns

def summarize_with_attention(sentence, model, dataset, device, max_len=100):
    model.eval()
    
    # 1. Tokenize và chuyển thành tensor
    if isinstance(sentence, str):
        tokens = dataset.tokenize(sentence)
    else:
        tokens = [token.lower() for token in sentence]
        
    tokens = tokens[:Config.MAX_LEN_SRC-2]
    src_indexes = [dataset.config.SOS_IDX] + [dataset.vocab.get(t, dataset.config.UNK_IDX) for t in tokens] + [dataset.config.EOS_IDX]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device) # [src_len, 1]

    # 2. Encode
    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src_tensor)

    # 3. Decode từng bước
    trg_indexes = [dataset.config.SOS_IDX]
    attention_matrix = []
    
    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            attn_weights = model.decoder.attention(hidden, encoder_outputs)
            attention_matrix.append(attn_weights.squeeze(0))
            output, hidden = model.decoder(trg_tensor, hidden, encoder_outputs)
            
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == dataset.config.EOS_IDX:
            break
    
    # 4. Convert IDs to Words
    trg_tokens = []
    idx_to_word = {v: k for k, v in dataset.vocab.items()}
    for i in trg_indexes:
        trg_tokens.append(idx_to_word.get(i, '<unk>'))
    
    src_tokens = [idx_to_word.get(i, '<unk>') for i in src_indexes]
    
    attention_matrix = torch.stack(attention_matrix)
    return src_tokens, trg_tokens[1:-1], attention_matrix.cpu().numpy()